In [26]:
import numpy as np
import pandas as pd
import time
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

from gensim.models import Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

data = fetch_20newsgroups(subset='all')
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

y_all = np.concatenate([y_train, y_test])

In [18]:
def run_kmeans(X, y, name):
    print(f"\n===== {name} =====")
    start_time = time.time()
    # нормализация
    X_norm = normalize(X)

    # PCA
    pca = PCA(n_components=50, random_state=42)
    X_reduced = pca.fit_transform(X_norm)

    # KMeans
    kmeans = KMeans(n_clusters=20, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_reduced)

    # метрики
    sil = silhouette_score(X_reduced, clusters)
    ari = adjusted_rand_score(y, clusters)
    
    end_time = time.time()
    dur = round(end_time-start_time,2)
    
    print("Silhouette:", sil)
    print("ARI:", ari)
    print("Time: ",dur)

    return sil, ari, dur
results = []

In [19]:
# Binary
vectorizer = CountVectorizer(binary=True, stop_words='english')
X_all = vectorizer.fit_transform(X)
sil, ari, dur = run_kmeans(X_all, y_all, "Binary")
results.append(["Binary", sil, ari, dur])


===== Binary =====
Silhouette: 0.051346255379974
ARI: -1.3262974296829683e-05
Time:  7.23


In [20]:
# Bag of Words
vectorizer = CountVectorizer(stop_words='english')
X_all = vectorizer.fit_transform(X)
sil, ari, dur = run_kmeans(X_all, y_all, "Bag of Words")
results.append(["Bag of Words", sil, ari, dur])


===== Bag of Words =====
Silhouette: 0.06972455474683069
ARI: -2.025730677261622e-05
Time:  6.72


In [21]:
# TF-IDF
vectorizer = TfidfVectorizer(stop_words='english')
X_all = vectorizer.fit_transform(X)
sil, ari, dur = run_kmeans(X_all, y_all, "TF-IDF")
results.append(["TF-IDF", sil, ari, dur])


===== TF-IDF =====
Silhouette: 0.08235146713107323
ARI: -0.0001707123026526442
Time:  7.43


In [22]:
# TF-IDF bigrams
vectorizer = TfidfVectorizer(ngram_range=(1,2), stop_words='english')
X_all = vectorizer.fit_transform(X)
sil, ari, dur = run_kmeans(X_all, y_all, "TF-IDF Bigrams")
results.append(["TF-IDF Bigrams", sil, ari, dur])


===== TF-IDF Bigrams =====
Silhouette: 0.0897491076649104
ARI: 1.9328632869051793e-05
Time:  21.39


In [27]:
# Word2Vec
tokenized = [text.split() for text in X]
w2v = Word2Vec(tokenized, vector_size=100, window=5, min_count=2)

def doc_vector(tokens):
    vectors = [w2v.wv[w] for w in tokens if w in w2v.wv]
    return np.mean(vectors, axis=0) if len(vectors) > 0 else np.zeros(100)


X_all = np.array([doc_vector(t) for t in tokenized])
sil, ari, dur = run_kmeans(X_all, y_all, "Word2Vec")
results.append(["Word2Vec", sil, ari, dur])


===== Word2Vec =====
Silhouette: 0.05170842260122299
ARI: -2.8470673504825465e-05
Time:  6.27


In [30]:
# Doc2Vec
tagged = [TaggedDocument(words=text.split(), tags=[i]) for i, text in enumerate(X)]
d2v = Doc2Vec(tagged, vector_size=100, epochs=20)

X_all = np.array([d2v.infer_vector(text.split()) for text in X])
sil, ari, dur = run_kmeans(X_all, y_all, "Doc2Vec")
results.append(["Doc2Vec", sil, ari, dur])


===== Doc2Vec =====
Silhouette: 0.030246777459979057
ARI: -8.733851011461981e-05
Time:  6.51


In [31]:
# BERT
X_train = np.load('../data/processed/train_bert_vectors.npy')
X_test = np.load('../data/processed/test_bert_vectors.npy')
X_all = np.vstack([X_train, X_test])
sil, ari, dur = run_kmeans(X_all, y_all, "BERT")
results.append(["BERT", sil, ari, dur])



===== BERT =====
Silhouette: 0.08386396616697311
ARI: 0.4393105598244274
Time:  10.06


In [34]:
# таблица результатов
results_df = pd.DataFrame(results, columns=["Method", "Silhouette", "ARI", "Time"]).sort_values(by = "ARI", ascending=False)

print("\n=== Итоговая таблица ===")
print(results_df)

# сохранить
import os
os.makedirs("../results/tables/", exist_ok=True)
results_df.to_csv("../results/tables/clustering_results.csv", index=False)

print("\nРезультаты сохранены в results/tables/clustering_results.csv")


=== Итоговая таблица ===
           Method  Silhouette       ARI   Time
6            BERT    0.083864  0.439311  10.06
3  TF-IDF Bigrams    0.089749  0.000019  21.39
0          Binary    0.051346 -0.000013   7.23
1    Bag of Words    0.069725 -0.000020   6.72
4        Word2Vec    0.051708 -0.000028   6.27
5         Doc2Vec    0.030247 -0.000087   6.51
2          TF-IDF    0.082351 -0.000171   7.43

Результаты сохранены в results/clustering_results.csv
